# 04 · Risk Maps & Fire Detection

The deliverable maps (risk surface, per-region counts) and the satellite
fire-detection CNN.

> Run `scripts/04_train_tabular.py` and `scripts/07_visualize.py`; for the CNN,
> `uv sync --extra cnn` then `scripts/03_patches.py` + `scripts/05_train_cnn.py`.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json, numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from wildfire.config import load_config
cfg = load_config()
surface = gpd.read_parquet(cfg.path_for('data_processed') / 'risk_surface.parquet')
ax = surface.plot(column='risk', cmap='inferno', figsize=(10, 9), legend=True,
                  legend_kwds={'label': 'modeled fire risk', 'shrink': 0.6})
ax.set_title('Oregon wildfire risk surface'); ax.set_axis_off()

## Interactive heatmap (folium)

Rendered inline; also saved to `outputs/maps/risk_heatmap.html`.

In [ ]:
from wildfire.viz.maps import interactive_heatmap
import folium
path = interactive_heatmap(surface, cfg.path_for('maps') / 'risk_heatmap.html')
folium.Map  # ensure folium import
from IPython.display import IFrame
IFrame(src=str(path), width='100%', height=500)

## Highest-risk ecoregions

In [ ]:
if 'ecoregion' in surface.columns:
    (surface.groupby('ecoregion')['risk'].mean().sort_values()
        .plot.barh(figsize=(8, 4), color='#c1440e'))
    plt.xlabel('mean risk'); plt.title('Mean modeled risk by ecoregion'); plt.tight_layout()

## Fire-detection CNN — the patches it learns from

RGB composites of fire vs no-fire patches. Fire patches carry a hot SWIR signature
and depressed NBR/NDVI.

In [ ]:
from wildfire.ingest.patches import load_patches
try:
    data = load_patches(cfg)
    X, y, ch = data['X'], data['y'], data['channels']
    rgb = [ch.index(b) for b in ('B4', 'B3', 'B2')]
    fig, axes = plt.subplots(2, 6, figsize=(14, 5))
    for row, label in enumerate([1, 0]):
        idx = np.where(y == label)[0][:6]
        for col, i in enumerate(idx):
            img = np.clip(X[i][rgb].transpose(1, 2, 0) * 2.5, 0, 1)
            axes[row, col].imshow(img); axes[row, col].set_axis_off()
        axes[row, 0].set_ylabel(['no-fire', 'fire'][label])
    axes[0,0].set_title('fire patches', loc='left'); axes[1,0].set_title('no-fire patches', loc='left')
    plt.tight_layout()
except FileNotFoundError:
    print('No patches yet — run scripts/03_patches.py --synthetic')

## CNN test performance (held-out spatial blocks)

In [ ]:
p = cfg.path_for('metrics') / 'cnn_metrics.json'
if p.exists():
    cnn = json.loads(p.read_text())
    print('backbone:', cnn['backbone'])
    print('test metrics:', {k: round(v, 3) for k, v in cnn['test_metrics'].items() if isinstance(v, (int, float))})
    hist = pd.DataFrame(cnn['history'])
    if len(hist):
        hist.plot(x='epoch', y=['train_loss', 'pr_auc'], secondary_y='pr_auc', figsize=(8, 4))
        plt.title('CNN training curve'); plt.tight_layout()
else:
    print('Train the CNN first: scripts/05_train_cnn.py')

---
That's the full system: a risk heatmap, per-region fire-count predictions, and a
satellite fire detector — all validated with leakage-free CV. See `RESULTS.md`.